In [1]:

from pathlib import Path

base = Path("/lakehouse/default/Files")

script = base / "etape_26A_unification_mobilite.py"

print("Script trouvé :", script.exists())

if not script.exists():
    print("\nRecherche dans les sous-dossiers :")
    for fichier in base.rglob("*26A*.py"):
        print(fichier)


In [2]:

exec(
    compile(
        script.read_text(encoding="utf-8"),
        str(script),
        "exec"
    )
)


In [1]:

from pathlib import Path

script = Path(
    "/lakehouse/default/Files/"
    "etape_26B_accessibilite_multimodale_360m.py"
)

print("Script trouvé :", script.exists())

if not script.exists():
    raise FileNotFoundError(
        "Importe le script à la racine de Files."
    )

exec(
    compile(
        script.read_text(encoding="utf-8"),
        str(script),
        "exec"
    )
)


In [3]:
%pip install h3 folium branca

In [4]:

from pathlib import Path

script = Path(
    "/lakehouse/default/Files/"
    "etape_27_carte_accessibilite_multimodale.py"
)

print("Script trouvé :", script.exists())

if not script.exists():
    raise FileNotFoundError(
        "Importe le script dans Files du Lakehouse."
    )

exec(
    compile(
        script.read_text(encoding="utf-8"),
        str(script),
        "exec"
    )
)


In [5]:

import h3
import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql.functions import pandas_udf

print("=== ETAPE 28 : ZONES PRIORITAIRES ===")

source = spark.table(
    "gold.france_access_mobility_proxy_200m"
)

# Calculer des hexagones plus fins
@pandas_udf("string")
def h3_r7(latitudes: pd.Series,
          longitudes: pd.Series) -> pd.Series:
    return pd.Series([
        h3.latlng_to_cell(float(lat), float(lon), 7)
        for lat, lon in zip(latitudes, longitudes)
    ])

base = (
    source
    .filter(
        F.col("p_lat").isNotNull()
        & F.col("p_lon").isNotNull()
    )
    .withColumn(
        "h3_r7",
        h3_r7(F.col("p_lat"), F.col("p_lon"))
    )
)

# Population ayant peu ou pas de mobilite
priorites = (
    base.groupBy("territoire", "h3_r7")
    .agg(
        F.count("*").alias("nb_carreaux"),
        F.sum("population").alias("population_totale"),

        F.sum(
            F.when(
                F.col("nb_options_proxy_360m") == 0,
                F.col("population")
            ).otherwise(0.0)
        ).alias("population_sans_option"),

        F.sum(
            F.when(
                ~F.col(
                    "au_moins_une_option_hors_gtfs_360m"
                ),
                F.col("population")
            ).otherwise(0.0)
        ).alias("population_sans_mobilite_partagee"),

        F.sum(
            F.when(
                F.col("nb_modes_proxy_360m") < 2,
                F.col("population")
            ).otherwise(0.0)
        ).alias("population_moins_deux_modes")
    )
    .withColumn(
        "taux_sans_option_pct",
        F.when(
            F.col("population_totale") > 0,
            F.round(
                100 * F.col("population_sans_option")
                / F.col("population_totale"),
                2
            )
        ).otherwise(0.0)
    )
)

# Sauvegarde pour les analyses et Power BI
table = "gold.france_zones_prioritaires_h3_r7"

(
    priorites.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table)
)

print("Table créée :", table)

print("\n=== HEXAGONES PAR TERRITOIRE ===")
display(
    spark.table(table)
    .groupBy("territoire")
    .count()
    .orderBy("territoire")
)

print("\n=== TOP 20 ZONES NON DESSERVIES ===")
display(
    spark.table(table)
    .filter(F.col("population_totale") >= 200)
    .orderBy(
        F.desc("population_sans_option"),
        F.desc("taux_sans_option_pct")
    )
    .limit(20)
)


In [6]:
%pip install "osmnx>=2.0,<3.0" networkx scipy folium

In [7]:

from pathlib import Path

script = Path(
    "/lakehouse/default/Files/"
    "etape_28_5min_reseau_pieton_osm.py"
)

print("Script trouvé :", script.exists())

if not script.exists():
    raise FileNotFoundError("Script absent du Lakehouse")

exec(
    compile(
        script.read_text(encoding="utf-8"),
        str(script),
        "exec"
    )
)
